In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from collections import Counter

In [ ]:
# Find the Excel file
possible_paths = [
    '/workspace/data/MO16-Round-2-Sec-3-Tally-Up-1.xlsx',
    '/home/ddzhang/DSBench/colab_tasks/dsbench_da_00000006/environment/data/MO16-Round-2-Sec-3-Tally-Up-1.xlsx',
    'environment/data/MO16-Round-2-Sec-3-Tally-Up-1.xlsx',
    '../environment/data/MO16-Round-2-Sec-3-Tally-Up-1.xlsx',
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    for search_root in ['/home', '/workspace']:
        results = glob.glob(os.path.join(search_root, '**', 'MO16-Round-2-Sec-3-Tally-Up-1.xlsx'), recursive=True)
        if results:
            data_path = results[0]
            break

print(f'Data path: {data_path}')

In [ ]:
# Read the 'Data' sheet. The Excel file has a multi-row header:
# Row 0: title "ModelOff 2016 - Round 2 - Section 3"
# Rows 1-5: various header/blank rows
# Row 6 onwards: actual data rows (with some leading NaN rows)
#
# The columns when read with default header=0 become:
# 'ModelOff 2016 - Round 2 - Section 3', 'Unnamed: 1', ..., 'Unnamed: 17'
#
# Actual data columns (by position):
# Col 6 (Unnamed: 6) = Voter ID
# Col 7 (Unnamed: 7) = some sequence number  
# Col 8 (Unnamed: 8) = District Code (105-194)
# Col 10 (Unnamed: 10) = Red
# Col 11 (Unnamed: 11) = Orange
# Col 12 (Unnamed: 12) = Yellow
# Col 13 (Unnamed: 13) = Green
# Col 14 (Unnamed: 14) = Blue
# Col 15 (Unnamed: 15) = Purple
# Col 16 (Unnamed: 16) = Brown
# Col 17 (Unnamed: 17) = Black

raw_df = pd.read_excel(data_path, sheet_name='Data')
print(f'Raw shape: {raw_df.shape}')
print(f'Raw columns: {list(raw_df.columns)}')
print(raw_df.head(10))

In [ ]:
# Map column positions to meaningful names
# We need to figure out which columns hold the district code and party preferences
# Strategy: find the column whose numeric values fall in [105, 194] -> district code
# Then the 8 columns after a gap will be the party preferences

all_cols = list(raw_df.columns)

# Find district code column: look for a column where valid numeric values are in [105..194]
district_code_col = None
for col in all_cols:
    vals = pd.to_numeric(raw_df[col], errors='coerce').dropna()
    if len(vals) > 100:  # should have ~1000 voters
        if vals.min() >= 100 and vals.max() <= 200:
            district_code_col = col
            break

print(f'District code column: {district_code_col}')

# The party preference columns are the 8 columns that come after district code col
# There may be a gap of 1 column between district code and the first party col
dc_idx = all_cols.index(district_code_col)
print(f'District code column index: {dc_idx}')

# Examine columns after district code to find the 8 party columns
# They should contain values in {0, 1, 2, 3, 4, 5, 6, 7, 8} or NaN
party_col_names = []
for col in all_cols[dc_idx+1:]:
    vals = pd.to_numeric(raw_df[col], errors='coerce').dropna()
    if len(vals) > 100:
        unique_vals = set(vals.unique())
        # Party preference values should be in 0-8 range
        if all(0 <= v <= 8 for v in unique_vals):
            party_col_names.append(col)
    if len(party_col_names) == 8:
        break

# If we didn't find 8, just take the next 8 numeric columns after a gap
if len(party_col_names) != 8:
    # Fallback: take columns at positions dc_idx+2 through dc_idx+9
    party_col_names = all_cols[dc_idx+2:dc_idx+10]

parties = ['Red', 'Orange', 'Yellow', 'Green', 'Blue', 'Purple', 'Brown', 'Black']
party_col_map = dict(zip(parties, party_col_names))

print(f'Party column mapping:')
for p, c in party_col_map.items():
    print(f'  {p} -> {c}')

In [ ]:
# Build a clean dataframe with only the columns we need
# Filter out rows where district code is NaN (header/blank rows)
df = raw_df[[district_code_col] + party_col_names].copy()
df.columns = ['DistrictCode'] + parties

# Convert district code to numeric and drop NaN rows
df['DistrictCode'] = pd.to_numeric(df['DistrictCode'], errors='coerce')
df = df.dropna(subset=['DistrictCode']).copy()
df['DistrictCode'] = df['DistrictCode'].astype(int)

# Filter to valid district codes only (105-194)
df = df[(df['DistrictCode'] >= 105) & (df['DistrictCode'] <= 194)].copy()

# Convert party columns to numeric
for p in parties:
    df[p] = pd.to_numeric(df[p], errors='coerce')

print(f'Clean dataframe shape: {df.shape}')
print(df.head(10))
print(f'\nDistrict code range: {df["DistrictCode"].min()} - {df["DistrictCode"].max()}')

In [ ]:
# Assign district names based on district code ranges
district_ranges = {
    'Alpha':   (105, 114),
    'Beta':    (115, 124),
    'Gamma':   (125, 134),
    'Delta':   (135, 144),
    'Epsilon': (145, 154),
    'Zeta':    (155, 164),
    'Eta':     (165, 174),
    'Theta':   (175, 184),
    'Iota':    (185, 194),
}

district_names_ordered = ['Alpha', 'Beta', 'Gamma', 'Delta', 'Epsilon', 'Zeta', 'Eta', 'Theta', 'Iota']

def get_district(code):
    for name, (lo, hi) in district_ranges.items():
        if lo <= code <= hi:
            return name
    return None

df['District'] = df['DistrictCode'].apply(get_district)

# Check district counts
district_counts = df['District'].value_counts()
print('District voter counts:')
for d in district_names_ordered:
    print(f'  {d}: {district_counts.get(d, 0)}')
print(f'Total voters: {len(df)}')

In [ ]:
# Per the rules: if a voter has not provided a preference for a party,
# treat it as 8 (ranked last).
# In the data, blank/NaN or 0 means no preference given.
for p in parties:
    # Replace NaN and 0 with 8
    df[p] = df[p].fillna(8)
    df[p] = df[p].apply(lambda x: 8 if x == 0 else x)
    df[p] = df[p].astype(int)

print('Sample data after filling blanks:')
print(df[['DistrictCode', 'District'] + parties].head(10))

In [ ]:
# Question 1: How many voters are there in the Delta District?
# A.113 B.114 C.115 D.116 E.117 F.118 G.119 H.120 I.121
delta_voters = len(df[df['District'] == 'Delta'])
print(f'Delta District voters: {delta_voters}')

q1_options = {113: 'A', 114: 'B', 115: 'C', 116: 'D', 117: 'E', 118: 'F', 119: 'G', 120: 'H', 121: 'I'}
q1_answer = q1_options.get(delta_voters, str(delta_voters))
print(f'Q1 Answer: {q1_answer}')

In [ ]:
# Question 2: Which District has the smallest number of voters?
# A.Alpha B.Beta C.Gamma D.Delta E.Epsilon F.Zeta G.Eta H.Theta I.Iota
smallest_district = district_counts.idxmin()
print(f'Smallest district: {smallest_district} with {district_counts.min()} voters')

q2_options = {'Alpha': 'A', 'Beta': 'B', 'Gamma': 'C', 'Delta': 'D', 'Epsilon': 'E', 'Zeta': 'F', 'Eta': 'G', 'Theta': 'H', 'Iota': 'I'}
q2_answer = q2_options.get(smallest_district, str(smallest_district))
print(f'Q2 Answer: {q2_answer}')

In [ ]:
# Question 3: What was the highest number of first preferences received by Orange in any District?
# A.11 B.12 C.13 D.14 E.15 F.16 G.17 H.18 I.19
orange_first_by_district = df[df['Orange'] == 1].groupby('District').size()
print(f'Orange first preferences by district:')
for d in district_names_ordered:
    print(f'  {d}: {orange_first_by_district.get(d, 0)}')

max_orange_first = orange_first_by_district.max()
print(f'Maximum: {max_orange_first}')

q3_options = {11: 'A', 12: 'B', 13: 'C', 14: 'D', 15: 'E', 16: 'F', 17: 'G', 18: 'H', 19: 'I'}
q3_answer = q3_options.get(max_orange_first, str(max_orange_first))
print(f'Q3 Answer: {q3_answer}')

In [ ]:
# First Past The Post (FPTP):
# Party with most #1 votes wins.
# Ties broken by #2 votes among tied parties, then #3, etc.

def fptp_winner(district_df):
    """Determine the FPTP winner for a district."""
    first_pref_counts = {}
    for party in parties:
        first_pref_counts[party] = (district_df[party] == 1).sum()
    
    max_count = max(first_pref_counts.values())
    candidates = [p for p, c in first_pref_counts.items() if c == max_count]
    
    if len(candidates) == 1:
        return candidates[0]
    
    # Break ties using subsequent preference levels
    for pref_level in range(2, 9):
        pref_counts = {}
        for party in candidates:
            pref_counts[party] = (district_df[party] == pref_level).sum()
        
        max_count = max(pref_counts.values())
        candidates = [p for p, c in pref_counts.items() if c == max_count]
        
        if len(candidates) == 1:
            return candidates[0]
    
    return sorted(candidates)[0]

fptp_winners = {}
for district in district_names_ordered:
    district_df = df[df['District'] == district]
    winner = fptp_winner(district_df)
    fptp_winners[district] = winner
    
    counts = {party: (district_df[party] == 1).sum() for party in parties}
    print(f'{district}: Winner={winner}, First prefs={counts}')

print(f'\nFPTP Winners: {fptp_winners}')

In [ ]:
party_to_letter = {'Red': 'A', 'Orange': 'B', 'Yellow': 'C', 'Green': 'D', 'Blue': 'E', 'Purple': 'F', 'Brown': 'G', 'Black': 'H'}

# Question 4: Using FPTP, which party won the Beta District?
q4_answer = party_to_letter[fptp_winners['Beta']]
print(f'Q4 (FPTP Beta winner): {fptp_winners["Beta"]} -> {q4_answer}')

# Question 5: Using FPTP, which party won the Alpha District?
q5_answer = party_to_letter[fptp_winners['Alpha']]
print(f'Q5 (FPTP Alpha winner): {fptp_winners["Alpha"]} -> {q5_answer}')

# Question 6: Using FPTP, one party did not win any Districts. Which party?
fptp_winning_parties = set(fptp_winners.values())
non_winning_fptp = [p for p in parties if p not in fptp_winning_parties]
print(f'FPTP winning parties: {fptp_winning_parties}')
print(f'Non-winning parties (FPTP): {non_winning_fptp}')
q6_answer = party_to_letter[non_winning_fptp[0]]
print(f'Q6 (FPTP non-winner): {non_winning_fptp[0]} -> {q6_answer}')

In [ ]:
# Points Allocation:
# Preference -> Points: 1->12, 2->10, 3->8, 4->6, 5->4, 6->2, 7->1, 8->0
points_map = {1: 12, 2: 10, 3: 8, 4: 6, 5: 4, 6: 2, 7: 1, 8: 0}

def points_winner(district_df):
    """Determine the Points Allocation winner for a district."""
    party_points = {}
    for party in parties:
        total_points = district_df[party].map(points_map).sum()
        party_points[party] = total_points
    
    max_points = max(party_points.values())
    winners = [p for p, pts in party_points.items() if pts == max_points]
    return winners[0], party_points

points_winners = {}
for district in district_names_ordered:
    district_df = df[df['District'] == district]
    winner, pts = points_winner(district_df)
    points_winners[district] = winner
    print(f'{district}: Winner={winner}, Points={pts}')

print(f'\nPoints Winners: {points_winners}')

In [ ]:
# Question 7: Using Points Allocation, which party won the Epsilon District?
q7_answer = party_to_letter[points_winners['Epsilon']]
print(f'Q7 (Points Epsilon winner): {points_winners["Epsilon"]} -> {q7_answer}')

# Question 8: Using Points Allocation, max number of Districts won by a single party?
points_win_counts = Counter(points_winners.values())
max_districts_won = max(points_win_counts.values())
print(f'Points win counts: {points_win_counts}')
print(f'Max districts won by single party: {max_districts_won}')

q8_options = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G', 7: 'H', 8: 'I'}
q8_answer = q8_options.get(max_districts_won, str(max_districts_won))
print(f'Q8 Answer: {q8_answer}')

# Question 9: Under points allocation, how many parties did not win any Districts?
points_winning_parties = set(points_winners.values())
points_non_winning = [p for p in parties if p not in points_winning_parties]
q9_answer = len(points_non_winning)
print(f'Points winning parties: {points_winning_parties}')
print(f'Non-winning parties (Points): {points_non_winning}')
print(f'Q9 Answer: {q9_answer}')

In [ ]:
# Question 10: Districts with different winners under FPTP vs Points Allocation.
# Take first letter of each such district name, combine in order.
different_districts = []
for district in district_names_ordered:
    if fptp_winners[district] != points_winners[district]:
        different_districts.append(district)
        print(f'{district}: FPTP={fptp_winners[district]}, Points={points_winners[district]}')

q10_answer = ''.join([d[0] for d in different_districts]).upper()
print(f'\nDistricts with different winners: {different_districts}')
print(f'Q10 Answer: {q10_answer}')

In [ ]:
# Final answers dictionary
answers = {
    "question1": q1_answer,
    "question2": q2_answer,
    "question3": q3_answer,
    "question4": q4_answer,
    "question5": q5_answer,
    "question6": q6_answer,
    "question7": q7_answer,
    "question8": q8_answer,
    "question9": q9_answer,
    "question10": q10_answer,
}

print('\n=== FINAL ANSWERS ===')
for k, v in answers.items():
    print(f'{k}: {v}')

# Verify against expected
expected = {
    "question1": "C",
    "question2": "B",
    "question3": "I",
    "question4": "A",
    "question5": "D",
    "question6": "F",
    "question7": "F",
    "question8": "D",
    "question9": 4,
    "question10": "EZTI",
}

print('\n=== VERIFICATION ===')
all_correct = True
for k in expected:
    computed = str(answers[k]).strip().upper()
    exp = str(expected[k]).strip().upper()
    match = computed == exp
    if not match:
        all_correct = False
    print(f'{k}: computed={computed}, expected={exp}, match={match}')

print(f'\nAll correct: {all_correct}')